# Feature-axis validation — is `design_name` really the right client axis?

Systematic re-check of the Phase-2/3 conclusion. Build size-invariant per-sample
descriptors, then run PCA + probes + variance decomposition against every metadata
factor. Whichever factor consistently explains the layout structure earns the FL
client axis; if something else (e.g. the `-a`/`-b` variant, macro geometry, or a
flow attribute in disguise) explains more, that's the honest partition axis.

## Predictions (frozen before running anything)

1. **PC1 ≈ macro / channel geometry** (macro-area fraction, distance-transform
   median of free space). Splits `-a` from `-b` almost perfectly.
2. **PC2 ≈ design scale**, ordering zero-riscy → RISCY → RISCY-FPU.
3. **Utilization** appears as a *smooth gradient*, not clusters.
4. **Power mesh** is near-invisible in routability features (would only show up
   in IR-drop channels, which the DRC feature set does not carry).
5. **Macro placement** sits between design and flow: enough spatial effect to
   move a point, not enough to redefine the cluster.

If any diagnostic contradicts the list, that is a finding — not a signal to
re-tune the pipeline.

## Workflow

1. Parse filenames into a factorial design matrix (already covered in the
   feature-analysis notebooks — repeated inline).
2. Stratified subsample: `N_PER_STRATUM` samples per
   (design × clock × utilization) cell so descriptor extraction stays cheap.
3. Build a ~80-descriptor vector per sample (distributional + textural +
   macro-geometry + anisotropy). QuantileTransform + StandardScaler.
4. PCA first, UMAP only as decoration. Recolor the same PC1/PC2 six times.
5. Quantify with kNN label agreement, probe AUC under GroupKFold, and
   variance-weighted η² decomposition on the leading PCs.
6. Two guards — die-area-alone baseline and label-shuffle control.
7. Tile-level bag-of-visual-words: 32 × 32 patches → k-means archetypes →
   per-design archetype histogram.
8. Final verdict cell: does `design_name` win every diagnostic, or does
   another factor take one?

# Imports

In [ ]:
import os, sys, warnings
from collections import Counter
from time import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm

from scipy import stats as scipy_stats
from scipy.ndimage import label as ndi_label, distance_transform_edt
from scipy.stats import skew as scipy_skew

from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True, 'grid.color': 'white', 'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5', 'figure.facecolor': 'white',
})

FEATURE_DIR = '../routability_ir_drop_prediction/training_set_N28/DRC/feature'
CHANNEL_NAMES = [
    'macro_region', 'cell_density',
    'RUDY_long', 'RUDY_short', 'RUDY_pin_long',
    'congestion_eGR_H', 'congestion_eGR_V',
    'congestion_GR_H',  'congestion_GR_V',
]
SEED = 42
N_PER_STRATUM = 12          # samples per (design x clock x util) cell
STRATUM_COLS  = ['design_name','clock_ns','utilization']
print('Imports OK.')

# Parse filenames → factorial design matrix

Adds two derived columns beyond the standard parser:

- `core` — the design family with the `-a`/`-b` suffix stripped
  (`RISCY-a` → `RISCY`).
- `variant` — the stripped suffix (`a`, `b`, or `n/a`).

This lets the probes measure the `-a` vs `-b` axis independently of the
design axis, which is exactly the split the physical-design engineer cares
about.

In [ ]:
def parse_sample_name(filename: str) -> dict:
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    for expected, token in zip(['c','u','m','p','f'], parts[-5:]):
        if not token.startswith(expected):
            raise ValueError(f'Bad token {token} in {filename}')
    c, u, m, p, f = parts[-5:]
    return {
        'design_name':      '-'.join(parts[:-6]),
        'macro_count':      parts[-6],
        'clock_ns':         float(c[1:]),
        'utilization':      float(u[1:]),
        'macro_placement':  m[1:],
        'power_mesh':       p[1:],
        'filler_insertion': f[1:],
        'filename':         filename,
    }

files = sorted(f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy'))
df_meta = pd.DataFrame([parse_sample_name(f) for f in files])
df_meta.loc[~df_meta['macro_placement'].isin(['1','2','3','4']), 'macro_placement'] = '1'

def _split_variant(dn):
    for suf in ('-a','-b'):
        if dn.endswith(suf):
            return dn[:-2], suf[1:]
    return dn, 'n/a'
cv = df_meta['design_name'].map(_split_variant)
df_meta['core']    = cv.map(lambda t: t[0])
df_meta['variant'] = cv.map(lambda t: t[1])

FACTOR_COLS = ['design_name','core','variant','macro_count','clock_ns',
               'utilization','macro_placement','power_mesh','filler_insertion']
print(f'Samples: {len(df_meta)}')
print('Unique-value counts per factor:')
for c in FACTOR_COLS:
    print(f'  {c:20s} {df_meta[c].nunique():3d}')
print()
print(df_meta[FACTOR_COLS].head())

# Stratified subsample

`N_PER_STRATUM` samples per (design × clock × utilization). Ensures every
cell of the design × persona grid is represented so the probes and
variance-decomposition are not confounded by unbalanced counts. On N28's
6 × 3 × 5 = 90 strata this gives ~1000 samples — enough for stable
descriptors, cheap enough to iterate.

In [ ]:
def _sample(g, k=N_PER_STRATUM, seed=SEED):
    return g.sample(n=min(k, len(g)), random_state=seed)

subset = (df_meta
          .groupby(STRATUM_COLS, group_keys=False)
          .apply(_sample)
          .reset_index(drop=True))
print(f'Subsample: {len(subset)} samples over '
      f'{subset[STRATUM_COLS].drop_duplicates().shape[0]} strata')
print(subset.groupby(STRATUM_COLS).size().describe())

# Descriptor builders

Each sample becomes a fixed-length vector, size-invariant by construction:

- **Per-channel distributional**: mean, std, skew, p50/p90/p99, hotspot
  fraction (share of tiles above the 90th percentile), Moran's I
  (spatial autocorrelation — blobby vs granular).
- **Macro geometry** (from `macro_region`): macro-area fraction, number of
  connected macro components, and the distance transform of *free space*
  reported as median / p10 / p90 — that p10 literally measures the
  narrowest routing channel and is where the `-a`/`-b` variant should show
  up.
- **Textural**: radially-averaged 2D power spectrum of `cell_density`
  (6 bins, normalized) — captures dominant spatial scale.
- **Anisotropy**: ratio of horizontal to vertical GR overflow — routing-
  direction stress is a real methodology fingerprint.
- **Die-area proxy**: pixel count. Used later as a one-feature control
  baseline (guard against a size-only confound)."

In [ ]:
def hotspot_frac(a, q=0.9):
    thr = np.quantile(a, q)
    return float((a > thr).mean())


def moran_i(a):
    """4-neighbour Moran's I on a 2-D array. In [-1, +1]."""
    a = a.astype(float)
    dev = a - a.mean()
    denom = (dev ** 2).sum() + 1e-12
    num = ((dev[:-1, :] * dev[1:, :]).sum()
           + (dev[:, :-1] * dev[:, 1:]).sum())
    n_pairs = (a.shape[0] - 1) * a.shape[1] + a.shape[0] * (a.shape[1] - 1)
    if n_pairs == 0:
        return 0.0
    return float((a.size * num) / (n_pairs * denom))


def radial_power_spectrum(a, n_bins=6):
    """Radially-averaged FFT power, normalized to a shape signature."""
    F = np.fft.fftshift(np.fft.fft2(a - a.mean()))
    P = np.abs(F) ** 2
    h, w = P.shape
    y, x = np.indices(P.shape)
    r = np.sqrt((y - h / 2) ** 2 + (x - w / 2) ** 2)
    r_max = min(h, w) / 2
    bins = np.linspace(0, r_max, n_bins + 1)
    profile = []
    for i in range(n_bins):
        m = (r >= bins[i]) & (r < bins[i + 1])
        profile.append(float(P[m].mean()) if m.any() else 0.0)
    total = sum(profile) + 1e-12
    return [p / total for p in profile]


def macro_geometry(macro_map, threshold=0.5):
    binary = macro_map > threshold
    labeled, n = ndi_label(binary)
    free = ~binary
    if free.any():
        dt = distance_transform_edt(free)
        dt_vals = dt[free]
        dt_med = float(np.median(dt_vals))
        dt_p10 = float(np.percentile(dt_vals, 10))
        dt_p90 = float(np.percentile(dt_vals, 90))
    else:
        dt_med = dt_p10 = dt_p90 = 0.0
    return {
        'macro_frac':         float(binary.mean()),
        'macro_n_components': int(n),
        'macro_dt_median':    dt_med,
        'macro_dt_p10':       dt_p10,
        'macro_dt_p90':       dt_p90,
    }


def channel_descriptors(a, name):
    a = a.astype(float)
    flat = a.ravel()
    return {
        f'{name}_mean':       float(flat.mean()),
        f'{name}_std':        float(flat.std()),
        f'{name}_skew':       float(scipy_skew(flat)),
        f'{name}_p50':        float(np.percentile(flat, 50)),
        f'{name}_p90':        float(np.percentile(flat, 90)),
        f'{name}_p99':        float(np.percentile(flat, 99)),
        f'{name}_hotspot90':  hotspot_frac(a, 0.9),
        f'{name}_moran':      moran_i(a),
    }


def build_descriptor(sample_path):
    arr = np.load(sample_path)
    if arr.ndim == 3 and arr.shape[0] < arr.shape[-1]:
        arr = np.transpose(arr, (1, 2, 0))    # normalize to (H, W, C)
    H, W, C = arr.shape
    out = {}
    for i, name in enumerate(CHANNEL_NAMES):
        if i < C:
            out.update(channel_descriptors(arr[:, :, i], name))
    if C > 0:
        out.update(macro_geometry(arr[:, :, 0]))
    if C > 1:
        prof = radial_power_spectrum(arr[:, :, 1], n_bins=6)
        for j, v in enumerate(prof):
            out[f'cd_radspec_{j}'] = v
    if C > 8:
        h_over = float(arr[:, :, 7].sum())
        v_over = float(arr[:, :, 8].sum())
        out['gr_h_over_v'] = h_over / (v_over + 1e-6)
    out['die_area_proxy'] = float(H * W)
    return out


approx = len(CHANNEL_NAMES) * 8 + 5 + 6 + 1 + 1
print(f'Descriptor vector length: ~{approx} '
      f'({len(CHANNEL_NAMES)} channels x 8 + 5 macro + 6 radspec + anisotropy + die_area)')

# Build the descriptor matrix

Runs `build_descriptor` over the stratified subset. Progress prints every
100 samples so you can bail if the timing is unreasonable. If a load
fails, the row is skipped and reported."

In [ ]:
t0 = time()
records = []
for i, row in enumerate(subset.itertuples()):
    path = os.path.join(FEATURE_DIR, row.filename)
    try:
        d = build_descriptor(path)
        d['filename'] = row.filename
        records.append(d)
    except Exception as e:
        print(f'  skip {row.filename}: {e}')
    if (i + 1) % 100 == 0:
        print(f'  {i + 1:4d}/{len(subset)}  elapsed={time() - t0:.1f}s')

df_desc = pd.DataFrame(records).set_index('filename')
df_desc = df_desc.replace([np.inf, -np.inf], np.nan)
df_desc = df_desc.fillna(df_desc.median(numeric_only=True))
print(f'\nDescriptor matrix: {df_desc.shape}   elapsed={time() - t0:.1f}s')

df_full = subset.set_index('filename').join(df_desc, how='inner')
DESC_COLS = df_desc.columns.tolist()
print(f'Joined table: {df_full.shape}   descriptor cols={len(DESC_COLS)}')

# Preprocess + PCA
`QuantileTransformer` on each column (heavy-tailed IR / congestion
distributions otherwise dominate every PC) followed by `StandardScaler`,
then a 10-component PCA. We keep the first 10 PCs regardless of
explained-variance elbow because the variance-decomposition below needs
the same fixed set of PCs across every factor to be comparable."

In [ ]:
X = df_full[DESC_COLS].to_numpy()

qt = QuantileTransformer(output_distribution='normal', random_state=SEED,
                         n_quantiles=min(1000, len(X)))
X_qt = qt.fit_transform(X)
sc = StandardScaler()
X_std = sc.fit_transform(X_qt)

pca = PCA(n_components=10, random_state=SEED)
Z = pca.fit_transform(X_std)
evr = pca.explained_variance_ratio_
print('Explained variance ratio:')
for i, v in enumerate(evr):
    print(f'  PC{i + 1:2d}: {v:.3f}   cumulative={evr[:i + 1].sum():.3f}')

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(range(1, 11), evr, alpha=0.85, color='#2c3e50', edgecolor='white')
ax.plot(range(1, 11), np.cumsum(evr), marker='o', color='#c0392b')
ax.set_xlabel('component'); ax.set_ylabel('explained variance ratio')
ax.set_title('PCA scree (10 components)')
plt.tight_layout(); plt.show()

# PC loadings + recolored PC1/PC2 scatter

First: print the top-10 descriptors by absolute loading on PC1/PC2/PC3.
If the story lines up with the prediction (PC1 dominated by macro-geometry
descriptors, PC2 by scale-like ones), the manifold learner is decoration.

Second: the same PC1/PC2 scatter recolored by six metadata factors. A
factor that produces visibly separated clusters in the PC1/PC2 view is
one that the descriptors encode directly."

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=DESC_COLS,
                        columns=[f'PC{i + 1}' for i in range(pca.n_components_)])

for pc in ['PC1', 'PC2', 'PC3']:
    top = loadings[pc].abs().sort_values(ascending=False).head(10)
    print(f'\nTop-10 |loadings| on {pc}:')
    for name in top.index:
        print(f'  {loadings.loc[name, pc]:+.3f}  {name}')

recolor_factors = ['design_name','core','variant','macro_placement',
                   'utilization','clock_ns','power_mesh']
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, factor in zip(axes.ravel(), recolor_factors):
    if pd.api.types.is_numeric_dtype(df_full[factor]):
        codes = df_full[factor].to_numpy()
        sc_ = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap='viridis',
                         s=12, alpha=0.75, edgecolor='none')
        plt.colorbar(sc_, ax=ax, fraction=0.04, label=factor)
    else:
        levels = sorted(df_full[factor].astype(str).unique())
        code_map = {l: i for i, l in enumerate(levels)}
        codes = df_full[factor].astype(str).map(code_map).to_numpy()
        cmap = mcm.get_cmap('tab10' if len(levels) <= 10 else 'tab20',
                            max(len(levels), 3))
        sc_ = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap=cmap,
                         s=12, alpha=0.75, edgecolor='none',
                         vmin=0, vmax=max(len(levels) - 1, 1))
        cbar = plt.colorbar(sc_, ax=ax, ticks=range(len(levels)), fraction=0.04)
        cbar.set_ticklabels(levels)
        cbar.ax.tick_params(labelsize=7)
    ax.set_title(f'PC1 vs PC2  colored by {factor}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

# Quantify — kNN label agreement (descriptor space)

For each factor, take a point's 15 nearest neighbours in the standardized
descriptor space and report the fraction that share its label. Compare to
chance = `1 / n_levels`. Robust, cluster-assumption-free. Ranking by
`lift = purity − chance` is the honest way to compare factors with
different numbers of levels."

In [ ]:
def knn_purity(X_, labels, k=15):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_)
    _, idx = nn.kneighbors(X_)
    idx = idx[:, 1:]  # drop self-neighbour
    agree = np.zeros(len(labels))
    for i in range(len(labels)):
        neigh = labels[idx[i]]
        agree[i] = (neigh == labels[i]).mean()
    return float(agree.mean())


purity_rows = []
for f in FACTOR_COLS:
    lab = df_full[f].astype(str).to_numpy()
    n_lvl = len(set(lab))
    if n_lvl < 2:
        continue
    chance = 1.0 / n_lvl
    p = knn_purity(X_std, lab, k=15)
    purity_rows.append({
        'factor':          f,
        'n_levels':        n_lvl,
        'chance':          round(chance, 3),
        'knn_purity@15':   round(p, 3),
        'lift':            round(p - chance, 3),
    })
purity_df = pd.DataFrame(purity_rows).sort_values('lift', ascending=False)
print(purity_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(purity_df['factor'], purity_df['lift'], color='#2980b9',
        alpha=0.9, edgecolor='white')
ax.invert_yaxis()
for i, v in enumerate(purity_df['lift']):
    ax.text(v + 0.005, i, f'{v:.2f}', va='center', fontsize=9)
ax.set_xlabel('lift = purity - chance')
ax.set_title('kNN(15) label agreement in descriptor space')
plt.tight_layout(); plt.show()

# Probe AUC under `GroupKFold`

Two probes per factor (logistic regression and gradient boosting) with
3-fold `GroupKFold`. `groups = design_name` prevents the trivial leakage\n",
where two samples of the same design end up in both train and test — that\n",
leakage would report ~0.99 AUC on every factor and mean nothing.\n",
\n",
For factors that **are** design (design_name / core / variant), a plain
design-group would leave a fold with only one class, so we fall back to\n",
grouping on `design_name × macro_placement` (~18 groups) instead. AUC is\n",
reported as macro-average one-vs-rest for multi-class targets."

In [ ]:
def probe_auc(X_, y, groups, model_name='logit', k=3):
    if model_name == 'logit':
        model = LogisticRegression(max_iter=1000, multi_class='ovr')
    else:
        model = GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                           random_state=SEED)
    n_groups = len(set(groups))
    k = min(k, n_groups)
    if k < 2:
        return float('nan'), float('nan')
    gkf = GroupKFold(n_splits=k)
    aucs = []
    for tr, te in gkf.split(X_, y, groups):
        try:
            model.fit(X_[tr], y[tr])
            proba = model.predict_proba(X_[te])
            classes = np.unique(y[tr])
            if len(classes) < 2 or len(np.unique(y[te])) < 2:
                continue
            if proba.shape[1] == 2:
                aucs.append(roc_auc_score(y[te], proba[:, 1]))
            else:
                aucs.append(roc_auc_score(y[te], proba, multi_class='ovr',
                                          average='macro', labels=classes))
        except Exception:
            continue
    if not aucs:
        return float('nan'), float('nan')
    return float(np.mean(aucs)), float(np.std(aucs))


groups_default = df_full['design_name'].astype(str).to_numpy()
groups_alt     = (df_full['design_name'].astype(str) + '|'
                  + df_full['macro_placement'].astype(str)).to_numpy()

probe_rows = []
for f in FACTOR_COLS:
    y = df_full[f].astype(str).to_numpy()
    if len(set(y)) < 2:
        continue
    groups = groups_alt if f in ('design_name', 'core', 'variant') else groups_default
    m_lg, s_lg = probe_auc(X_std, y, groups, 'logit', k=3)
    m_gb, s_gb = probe_auc(X_std, y, groups, 'gbm',   k=3)
    probe_rows.append({
        'factor':    f,
        'n_levels':  len(set(y)),
        'logit_AUC': round(m_lg, 3), 'logit_std': round(s_lg, 3),
        'gbm_AUC':   round(m_gb, 3), 'gbm_std':   round(s_gb, 3),
    })
probe_df = pd.DataFrame(probe_rows).sort_values('gbm_AUC', ascending=False)
print(probe_df.to_string(index=False))

# Variance decomposition — variance-weighted η² on the leading PCs

Instead of PERMANOVA (which needs an n × n distance matrix), we compute
per-PC η² of each factor via one-way ANOVA and sum the contributions
weighted by that PC's explained variance. The `weighted_total` column\n",
approximates *fraction of layout variance explained by that factor*.\n",
Balanced design ⇒ these numbers can be read semi-quantitatively.\n",
\n",
This is the single most informative output for the client-axis question:\n",
whichever factor tops `weighted_total` is the axis the descriptors are\n",
actually structured by."

In [ ]:
def eta_sq(y_num, groups):
    d = pd.DataFrame({'y': y_num, 'g': groups}).dropna()
    grand = d['y'].mean()
    ss_tot = ((d['y'] - grand) ** 2).sum()
    if ss_tot == 0:
        return 0.0
    ss_bet = sum(len(g) * (g['y'].mean() - grand) ** 2
                 for _, g in d.groupby('g'))
    return float(ss_bet / ss_tot)


K_PC = pca.n_components_
eta_rows = []
for f in FACTOR_COLS:
    if df_full[f].nunique() < 2:
        continue
    groups = df_full[f].astype(str).to_numpy()
    row = {'factor': f, 'n_levels': int(df_full[f].nunique())}
    for k in range(K_PC):
        row[f'eta2_PC{k + 1}'] = round(eta_sq(Z[:, k], groups), 3)
    row['weighted_total'] = round(sum(
        row[f'eta2_PC{k + 1}'] * evr[k] for k in range(K_PC)
    ), 3)
    eta_rows.append(row)
eta_df = pd.DataFrame(eta_rows).sort_values('weighted_total', ascending=False)
print(eta_df.to_string(index=False))

top_factors = eta_df.head(6)['factor'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(top_factors))
cmap = mcm.get_cmap('viridis', K_PC)
for k in range(K_PC):
    heights = np.array([
        eta_df.set_index('factor').loc[f, f'eta2_PC{k + 1}'] * evr[k]
        for f in top_factors
    ])
    ax.bar(top_factors, heights, bottom=bottom, color=cmap(k),
           label=f'PC{k + 1}  (evr={evr[k]:.2f})',
           edgecolor='white', alpha=0.9)
    bottom += heights
ax.set_ylabel('variance-weighted eta^2')
ax.set_title('Fraction of layout variance explained by each factor')
ax.legend(fontsize=7, loc='upper right', ncol=2)
plt.xticks(rotation=25, ha='right')
plt.tight_layout(); plt.show()

# Guards

Two negative controls without which every AUC number above is
meaningless.

- **Die-area alone**: refits the probes using only the single
  `die_area_proxy` feature. If it gets close to the full-descriptor AUC
  for `variant` or `core`, the descriptor pipeline is measuring size, not
  physics — every downstream claim collapses to \"bigger designs cluster
  together.\"\n",
- **Label-shuffle**: permutes labels for the two most-separable factors\n",
  and refits. Must collapse to chance (~1/n_classes). If it doesn't,\n",
  either the group-splitting is broken or the AUC estimator is."

In [ ]:
print('Guard 1 — die-area alone (should be low if descriptors are not size-driven):')
X_area = df_full[['die_area_proxy']].to_numpy()
for f in ['design_name', 'core', 'variant', 'utilization', 'clock_ns']:
    y = df_full[f].astype(str).to_numpy()
    if len(set(y)) < 2:
        continue
    groups = groups_alt if f in ('design_name', 'core', 'variant') else groups_default
    m, _ = probe_auc(X_area, y, groups, 'gbm', k=3)
    print(f'  die_area alone -> AUC({f:15s}) = {m:.3f}')

print('\nGuard 2 — label-shuffle control (should collapse to chance):')
rng2 = np.random.default_rng(SEED + 7)
for f in ['design_name', 'variant', 'utilization']:
    y = df_full[f].astype(str).to_numpy()
    y_perm = rng2.permutation(y)
    groups = groups_alt if f in ('design_name', 'core', 'variant') else groups_default
    m, _ = probe_auc(X_std, y_perm, groups, 'gbm', k=3)
    print(f'  shuffled {f:15s} AUC = {m:.3f}   chance = {1.0 / len(set(y)):.3f}')

# Tile-level archetype map (bag-of-visual-words)

Independent view of the same question at tile granularity: sample
`TILES_PER_SAMPLE` random 32 × 32 patches per layout, describe each by
local density, RUDY, macro fraction and distance-to-nearest-macro, then
k-means into `N_TILE_ARCHETYPES` tile archetypes (macro interior,
macro-edge channel, dense logic core, sparse periphery, ...).

Each design is then a histogram over archetypes. If two designs (say
`RISCY-a` and `RISCY-b`) share a `core` but sit at different points in
archetype space, that's the concrete engineering statement — \"design B is\n",
31% channel tiles versus 4%\" — that the numeric probes only imply."

In [ ]:
TILE_SIZE          = 32
TILES_PER_SAMPLE   = 12
N_TILE_ARCHETYPES  = 10
rng_t = np.random.default_rng(SEED + 11)


def sample_tile_descs(arr, k=TILES_PER_SAMPLE, s=TILE_SIZE):
    H, W, C = arr.shape
    if H < s or W < s:
        return []
    macro = arr[:, :, 0] > 0.5
    dt = distance_transform_edt(~macro) if (~macro).any() else np.zeros_like(macro, dtype=float)
    out = []
    ys = rng_t.integers(0, H - s + 1, size=k)
    xs = rng_t.integers(0, W - s + 1, size=k)
    for y, x in zip(ys, xs):
        patch = arr[y:y + s, x:x + s, :]
        cd    = patch[:, :, 1]
        rudy  = patch[:, :, 2] if C > 2 else cd
        m_frac = float((patch[:, :, 0] > 0.5).mean())
        dt_patch = dt[y:y + s, x:x + s]
        out.append([
            float(cd.mean()), float(cd.std()),
            float(rudy.mean()), float(rudy.std()),
            m_frac,
            float(np.median(dt_patch)),
        ])
    return out


tile_recs, tile_owner = [], []
for row in subset.itertuples():
    try:
        arr = np.load(os.path.join(FEATURE_DIR, row.filename))
        if arr.ndim == 3 and arr.shape[0] < arr.shape[-1]:
            arr = np.transpose(arr, (1, 2, 0))
        for d in sample_tile_descs(arr):
            tile_recs.append(d)
            tile_owner.append(row.filename)
    except Exception:
        pass

tile_X = np.array(tile_recs)
tile_owner = np.array(tile_owner)
print(f'Tile descriptors: {tile_X.shape}')

tile_std = StandardScaler().fit_transform(tile_X)
km = KMeans(n_clusters=N_TILE_ARCHETYPES, n_init=10, random_state=SEED)
tile_labels = km.fit_predict(tile_std)

hist_rows = []
for design, group_files in df_full.reset_index().groupby('design_name')['filename']:
    m = np.isin(tile_owner, group_files.to_numpy())
    if m.sum() == 0:
        continue
    counts = Counter(tile_labels[m])
    total = sum(counts.values())
    row = {'design_name': design}
    for k in range(N_TILE_ARCHETYPES):
        row[f'arch_{k}'] = round(counts.get(k, 0) / total, 3)
    hist_rows.append(row)
hist_df = pd.DataFrame(hist_rows).set_index('design_name')
print(hist_df.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(hist_df.values, cmap='YlOrRd', aspect='auto',
               vmin=0, vmax=hist_df.values.max())
ax.set_yticks(range(hist_df.shape[0]))
ax.set_yticklabels(hist_df.index)
ax.set_xticks(range(hist_df.shape[1]))
ax.set_xticklabels(hist_df.columns, rotation=0)
for i in range(hist_df.shape[0]):
    for j in range(hist_df.shape[1]):
        v = hist_df.values[i, j]
        if v >= 0.02:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if v > 0.35 else 'black')
ax.set_title('Design × tile-archetype composition (row-normalized)')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.show()

# Verdict — decide the client axis from the diagnostics

Read the four diagnostics *together* — no single number decides:\n",
\n",
1. **Variance decomposition** (`eta_df.weighted_total`) — fraction of\n",
   layout variance the factor explains. This is the primary evidence.\n",
2. **Probe AUC under GroupKFold** — how learnable the factor is from the\n",
   descriptors, with leakage suppressed.\n",
3. **kNN purity lift** — nearest-neighbour agreement in descriptor space.\n",
4. **Tile-archetype histogram** — engineering-grade sanity check: does\n",
   the ranking of designs on the archetype axes match the ranking of\n",
   designs on the numeric factor?\n",
\n",
Decision rule (write the ranking BEFORE reading the guards):\n",
\n",
- **`design_name` wins if** it tops variance decomposition AND either\n",
  probe AUC or kNN lift, AND the die-area-alone AUC on `variant`/`core`\n",
  is well below the full-descriptor probe AUC (guard 1 passes), AND\n",
  the shuffle control collapses to chance (guard 2 passes).\n",
- **Switch to a finer axis** (e.g. `variant`, or `design_name × variant`)\n",
  **if** `variant` gets a comparable or higher weighted-η² AND\n",
  survives both guards. That would mean the `-a`/`-b` split is the\n",
  physical signal and `design_name` was only a coarser proxy.\n",
- **Switch to a coarser axis** (e.g. `core`) if `variant` is invisible\n",
  and `core` explains most of the variance — indicates the `-a`/`-b`\n",
  variants are essentially interchangeable in feature space.\n",
- **Drop the physical-design axis entirely** if a flow attribute\n",
  (`macro_placement`, `power_mesh`) explains more of the variance than\n",
  any design column — that would flip Phase 2's ownership decision.\n",
\n",
Any of those outcomes is a valid answer; the point of this notebook is\n",
that you commit to the rule before looking at the plots."